# Grafos e representação de grafos — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula e segue de perto o material
[*Algoritmos para Grafos (em linguagem C)*](https://www.ime.usp.br/~pf/algoritmos_para_grafos/),
de Paulo Feofiloff (IME-USP), nos capítulos
[Grafos](https://www.ime.usp.br/~pf/algoritmos_para_grafos/aulas/graphs.html) e
[Estruturas de dados para grafos](https://www.ime.usp.br/~pf/algoritmos_para_grafos/aulas/graphdatastructs.html).

## Objetivos

Ao final deste tutorial você será capaz de:

- reconhecer os elementos de um grafo (vértices, arcos, pontas, vizinhança) em código C;
- implementar a representação por **matriz de adjacências** e suas operações básicas;
- implementar a representação por **listas de adjacência** e suas operações básicas;
- calcular graus de entrada e de saída, e identificar fontes e sorvedouros;
- converter uma representação na outra e justificar a escolha entre elas;
- construir grafos **não-dirigidos** a partir de arestas.

## Como este notebook funciona

O kernel é Python 3, mas **todo o código do tutorial é C**. Cada exemplo é
gravado em um arquivo `.c` com a mágica `%%writefile` e depois compilado e
executado com `gcc`. Execute as células na ordem.

In [ ]:
# Preparação do ambiente: cria a pasta de trabalho e confere o compilador.
import os, subprocess, textwrap

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

def compilar_e_rodar(fonte, entrada=None):
    \"\"\"Compila src/<fonte> com gcc (C99) e executa, mostrando a saída.\"\"\"
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    c = subprocess.run(['gcc', '-std=c99', '-Wall', os.path.join('src', fonte), '-o', exe],
                       capture_output=True, text=True)
    if c.returncode != 0:
        print('ERRO DE COMPILACAO:\n' + c.stderr)
        return
    if c.stderr:
        print('avisos do compilador:\n' + c.stderr)
    r = subprocess.run([exe], capture_output=True, text=True, input=entrada)
    print(r.stdout, end='')
    if r.stderr:
        print('stderr:', r.stderr)

## 1. Representação por matriz de adjacências

A **matriz de adjacências** de um grafo é uma matriz booleana com linhas e
colunas indexadas pelos vértices: `adj[v][w] = 1` se `v-w` é um arco, e `0` caso
contrário. A linha `v` representa o *leque de saída* de `v`; a coluna `w`
representa o *leque de entrada* de `w`.

Como nossos grafos não têm laços, a diagonal é nula. O espaço ocupado é
proporcional a $V^2$ — apropriado para grafos **densos**.

O grafo usado como exemplo aqui é o mesmo da aula, com arcos
`0-1  0-5  1-0  1-5  2-4  3-1  5-3`.

In [ ]:
%%writefile src/grafo_matriz.c
/* REPRESENTACAO POR MATRIZ DE ADJACENCIAS
   Adaptado de P. Feofiloff, "Algoritmos para Grafos (em linguagem C)", IME-USP.
   https://www.ime.usp.br/~pf/algoritmos_para_grafos/aulas/graphdatastructs.html */
#include <stdio.h>
#include <stdlib.h>

#define vertex int

struct graph {
   int V;      /* numero de vertices */
   int A;      /* numero de arcos    */
   int **adj;  /* matriz de adjacencias */
};
typedef struct graph *Graph;

/* Aloca matriz r x c com todos os elementos iguais a val. */
static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i)
      m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j)
         m[i][j] = val;
   return m;
}

/* Constroi um grafo com vertices 0 1 .. V-1 e nenhum arco. */
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V;
   G->A = 0;
   G->adj = MATRIXint( V, V, 0);
   return G;
}

/* Insere o arco v-w. Se ele ja existe, nao faz nada. */
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) {
      G->adj[v][w] = 1;
      G->A++;
   }
}

/* Remove o arco v-w. Se ele nao existe, nao faz nada. */
void GRAPHremoveArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 1) {
      G->adj[v][w] = 0;
      G->A--;
   }
}

/* Imprime, para cada vertice v, a lista de seus vizinhos. */
void GRAPHshow( Graph G) {
   for (vertex v = 0; v < G->V; ++v) {
      printf( "%2d:", v);
      for (vertex w = 0; w < G->V; ++w)
         if (G->adj[v][w] == 1)
            printf( " %2d", w);
      printf( "\n");
   }
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1);
   GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0);
   GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4);
   GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);
   GRAPHinsertArc( G, 5, 3);   /* arco repetido: nao deve alterar A */

   printf( "V = %d, A = %d, tamanho = %d\n", G->V, G->A, G->V + G->A);
   printf( "matriz de adjacencias:\n    ");
   for (vertex w = 0; w < G->V; ++w) printf( "%2d", w);
   printf( "\n");
   for (vertex v = 0; v < G->V; ++v) {
      printf( "%2d: ", v);
      for (vertex w = 0; w < G->V; ++w) printf( "%2d", G->adj[v][w]);
      printf( "\n");
   }
   printf( "vizinhos:\n");
   GRAPHshow( G);
   return 0;
}

In [ ]:
compilar_e_rodar('grafo_matriz.c')
# Saida esperada: V = 6, A = 7, tamanho = 13; linha 4 nula (sorvedouro);
# coluna 2 nula (fonte).

### Exercício 1 — graus de entrada e de saída (matriz)

Escreva `GRAPHoutdeg(G, v)`, que devolve o grau de saída de `v`, e
`GRAPHindeg(G, v)`, que devolve o grau de entrada de `v`.

Lembre-se: o grau de saída de `v` é o número de `1`s na **linha** `v`; o grau de
entrada é o número de `1`s na **coluna** `v`.

*(Exercício adaptado de Feofiloff, cap. Estruturas de dados para grafos, Ex. 1 —
Sedgewick 17.40.)*

In [ ]:
%%writefile src/ex1_graus.c
/* Exercicio 1: graus na representacao por matriz de adjacencias. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

/* TODO: devolva o numero de arcos que SAEM de v. */
int GRAPHoutdeg( Graph G, vertex v) {
   /* implemente aqui */
   return -1;
}

/* TODO: devolva o numero de arcos que ENTRAM em v. */
int GRAPHindeg( Graph G, vertex v) {
   /* implemente aqui */
   return -1;
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);

   /* Testes automaticos: graus esperados do grafo da aula. */
   int outdeg[] = {2, 2, 1, 1, 0, 1};
   int indeg[]  = {1, 2, 0, 1, 1, 2};
   int soma_out = 0, soma_in = 0, ok = 1;
   for (vertex v = 0; v < G->V; ++v) {
      int o = GRAPHoutdeg( G, v), i = GRAPHindeg( G, v);
      printf( "v=%d  outdeg=%d (esperado %d)  indeg=%d (esperado %d)\n",
              v, o, outdeg[v], i, indeg[v]);
      if (o != outdeg[v] || i != indeg[v]) ok = 0;
      soma_out += o; soma_in += i;
   }
   printf( "soma dos graus de saida = %d, de entrada = %d, A = %d\n",
           soma_out, soma_in, G->A);
   printf( ok && soma_out == G->A && soma_in == G->A
           ? "OK: verifique sua implementacao -- passou\n"
           : "FALHOU: revise sua implementacao\n");
   return 0;
}

In [ ]:
compilar_e_rodar('ex1_graus.c')

## 2. Representação por listas de adjacência

O **vetor de listas de adjacência** tem uma lista encadeada associada a cada
vértice; a lista de `v` contém todos os vizinhos de `v` e portanto representa o
*leque de saída* de `v`.

O espaço ocupado é proporcional a $V + A$, ou seja, ao **tamanho** do grafo —
muito mais econômico para grafos **esparsos**.

Repare em dois detalhes de `GRAPHinsertArc()`:

1. ela percorre a lista para não inserir um arco repetido — por isso, no pior
   caso, consome tempo proporcional ao número de arcos;
2. o novo nó entra no **início** da lista, de modo que a ordem impressa é o
   inverso da ordem de inserção.

In [ ]:
%%writefile src/grafo_listas.c
/* REPRESENTACAO POR LISTAS DE ADJACENCIA
   Adaptado de P. Feofiloff, "Algoritmos para Grafos (em linguagem C)", IME-USP. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int

typedef struct node *link;
struct node {
   vertex w;    /* vizinho */
   link next;   /* proximo no da lista */
};

struct graph {
   int V;
   int A;
   link *adj;   /* vetor de listas de adjacencia */
};
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));
   a->w = w;
   a->next = next;
   return a;
}

Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V;
   G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v)
      G->adj[v] = NULL;
   return G;
}

/* Insere o arco v-w. Se ele ja existe, nao faz nada.
   No pior caso consome tempo proporcional ao numero de arcos. */
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return;
   G->adj[v] = NEWnode( w, G->adj[v]);
   G->A++;
}

/* Versao de GRAPHshow() para listas de adjacencia. */
void GRAPHshow( Graph G) {
   for (vertex v = 0; v < G->V; ++v) {
      printf( "%2d:", v);
      for (link a = G->adj[v]; a != NULL; a = a->next)
         printf( " %2d", a->w);
      printf( "\n");
   }
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1);
   GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0);
   GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4);
   GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);

   printf( "V = %d, A = %d, tamanho = %d\n", G->V, G->A, G->V + G->A);
   printf( "vetor de listas de adjacencia:\n");
   GRAPHshow( G);
   return 0;
}

In [ ]:
compilar_e_rodar('grafo_listas.c')
# Saida esperada (note a ordem invertida dentro de cada lista):
#  0:  5  1
#  1:  5  0
#  2:  4
#  3:  1
#  4:
#  5:  3

### Exercício 2 — remoção de arco em listas de adjacência

Escreva `GRAPHremoveArc(G, v, w)`, que remove o arco `v-w` de um grafo
representado por listas de adjacência. A função deve liberar o nó removido com
`free()` e decrementar `G->A`. Se o arco não existe, ela não faz nada.

Atenção ao caso em que o nó a remover é o **primeiro** da lista.

*(Exercício adaptado de Feofiloff, cap. Estruturas de dados para grafos, Ex. 2.)*

In [ ]:
%%writefile src/ex2_remove.c
/* Exercicio 2: remocao de arco em listas de adjacencia. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));
   a->w = w; a->next = next;
   return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return;
   G->adj[v] = NEWnode( w, G->adj[v]);
   G->A++;
}
/* Devolve 1 se v-w e arco de G, e 0 em caso contrario. */
int GRAPHhasArc( Graph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return 1;
   return 0;
}

/* TODO: remova o arco v-w da lista de v, liberando o no e ajustando G->A. */
void GRAPHremoveArc( Graph G, vertex v, vertex w) {
   /* implemente aqui */
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);
   printf( "antes:  A = %d\n", G->A);

   GRAPHremoveArc( G, 0, 5);   /* no do meio/fim da lista de 0 */
   GRAPHremoveArc( G, 1, 5);   /* primeiro no da lista de 1    */
   GRAPHremoveArc( G, 4, 2);   /* arco inexistente: nao faz nada */
   printf( "depois: A = %d\n", G->A);

   int ok = (G->A == 5)
         && !GRAPHhasArc( G, 0, 5) && GRAPHhasArc( G, 0, 1)
         && !GRAPHhasArc( G, 1, 5) && GRAPHhasArc( G, 1, 0);
   printf( ok ? "OK: verifique sua implementacao -- passou\n"
              : "FALHOU: revise sua implementacao\n");
   return 0;
}

In [ ]:
compilar_e_rodar('ex2_remove.c')

## 3. Fontes, sorvedouros e vértices isolados

- **fonte**: grau de entrada nulo — nenhuma coluna aponta para ele;
- **sorvedouro**: grau de saída nulo — sua lista de adjacência é vazia;
- **isolado**: os dois graus nulos.

No grafo do exemplo, `2` é fonte e `4` é sorvedouro.

### Exercício 3 — vetor `isSink[]`

Preencha `GRAPHsinks(G, isSink)` de modo que `isSink[v]` valha `1` se, e somente
se, `v` é sorvedouro. Use a representação por listas: um vértice é sorvedouro
exatamente quando sua lista é vazia.

*(Exercício adaptado de Feofiloff, cap. Estruturas de dados para grafos, Ex. 1 —
"Fontes e sorvedouros".)*

In [ ]:
%%writefile src/ex3_sinks.c
/* Exercicio 3: identificacao de sorvedouros (listas de adjacencia). */
#include <stdio.h>
#include <stdlib.h>

#define vertex int

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));
   a->w = w; a->next = next; return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return;
   G->adj[v] = NEWnode( w, G->adj[v]); G->A++;
}

/* TODO: isSink[v] = 1 se v e sorvedouro, 0 caso contrario. */
void GRAPHsinks( Graph G, int isSink[]) {
   /* implemente aqui */
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);

   int isSink[6];
   for (vertex v = 0; v < 6; ++v) isSink[v] = -1;
   GRAPHsinks( G, isSink);

   int esperado[] = {0, 0, 0, 0, 1, 0};   /* so o vertice 4 e sorvedouro */
   int ok = 1;
   for (vertex v = 0; v < 6; ++v) {
      printf( "isSink[%d] = %d (esperado %d)\n", v, isSink[v], esperado[v]);
      if (isSink[v] != esperado[v]) ok = 0;
   }
   printf( ok ? "OK: verifique sua implementacao -- passou\n"
              : "FALHOU: revise sua implementacao\n");
   return 0;
}

In [ ]:
compilar_e_rodar('ex3_sinks.c')

## 4. Convertendo uma representação na outra

Escolher a representação não é uma decisão irreversível: dá para converter uma
na outra. A conversão evidencia o custo de cada estrutura — varrer a matriz
inteira custa tempo proporcional a $V^2$, mesmo que o grafo tenha poucos arcos.

O exemplo abaixo mostra a conversão **matriz → listas**; o exercício pede o
caminho inverso.

In [ ]:
%%writefile src/conversao.c
/* Conversao matriz de adjacencias -> listas de adjacencia. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int

/* --- matriz --- */
struct mgraph { int V; int A; int **adj; };
typedef struct mgraph *MGraph;

MGraph MGRAPHinit( int V) {
   MGraph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (int *));
   for (vertex i = 0; i < V; ++i) {
      G->adj[i] = malloc( V * sizeof (int));
      for (vertex j = 0; j < V; ++j) G->adj[i][j] = 0;
   }
   return G;
}
void MGRAPHinsertArc( MGraph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

/* --- listas --- */
typedef struct node *link;
struct node { vertex w; link next; };
struct lgraph { int V; int A; link *adj; };
typedef struct lgraph *LGraph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));
   a->w = w; a->next = next; return a;
}
LGraph LGRAPHinit( int V) {
   LGraph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void LGRAPHinsertArc( LGraph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return;
   G->adj[v] = NEWnode( w, G->adj[v]); G->A++;
}

/* Percorre a matriz inteira: tempo proporcional a V*V. */
LGraph MATRIXtoLISTS( MGraph G) {
   LGraph H = LGRAPHinit( G->V);
   for (vertex v = 0; v < G->V; ++v)
      for (vertex w = G->V - 1; w >= 0; --w)   /* de tras pra frente:
                                                  preserva a ordem crescente */
         if (G->adj[v][w] == 1)
            LGRAPHinsertArc( H, v, w);
   return H;
}

int main( void) {
   MGraph G = MGRAPHinit( 6);
   MGRAPHinsertArc( G, 0, 1); MGRAPHinsertArc( G, 0, 5);
   MGRAPHinsertArc( G, 1, 0); MGRAPHinsertArc( G, 1, 5);
   MGRAPHinsertArc( G, 2, 4); MGRAPHinsertArc( G, 3, 1);
   MGRAPHinsertArc( G, 5, 3);

   LGraph H = MATRIXtoLISTS( G);
   printf( "matriz: V = %d, A = %d\n", G->V, G->A);
   printf( "listas: V = %d, A = %d\n", H->V, H->A);
   for (vertex v = 0; v < H->V; ++v) {
      printf( "%2d:", v);
      for (link a = H->adj[v]; a != NULL; a = a->next) printf( " %2d", a->w);
      printf( "\n");
   }
   return 0;
}

In [ ]:
compilar_e_rodar('conversao.c')

### Exercício 4 — conversão listas → matriz

Complete `LISTStoMATRIX()` no arquivo abaixo. Qual das duas conversões é mais
rápida para um grafo esparso? Justifique em uma frase na célula de resposta ao
final.

*(Exercício adaptado de Feofiloff, cap. Estruturas de dados para grafos, Ex. 1 —
"Transformação de uma representação em outra".)*

In [ ]:
%%writefile src/ex4_conversao.c
/* Exercicio 4: conversao listas de adjacencia -> matriz de adjacencias. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int

typedef struct node *link;
struct node { vertex w; link next; };
struct lgraph { int V; int A; link *adj; };
typedef struct lgraph *LGraph;
struct mgraph { int V; int A; int **adj; };
typedef struct mgraph *MGraph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));
   a->w = w; a->next = next; return a;
}
LGraph LGRAPHinit( int V) {
   LGraph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void LGRAPHinsertArc( LGraph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return;
   G->adj[v] = NEWnode( w, G->adj[v]); G->A++;
}
MGraph MGRAPHinit( int V) {
   MGraph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (int *));
   for (vertex i = 0; i < V; ++i) {
      G->adj[i] = malloc( V * sizeof (int));
      for (vertex j = 0; j < V; ++j) G->adj[i][j] = 0;
   }
   return G;
}

/* TODO: construa e devolva a representacao por matriz do grafo G.
   Nao se esqueca de manter o campo A coerente. */
MGraph LISTStoMATRIX( LGraph G) {
   MGraph H = MGRAPHinit( G->V);
   /* implemente aqui */
   return H;
}

int main( void) {
   LGraph G = LGRAPHinit( 6);
   LGRAPHinsertArc( G, 0, 1); LGRAPHinsertArc( G, 0, 5);
   LGRAPHinsertArc( G, 1, 0); LGRAPHinsertArc( G, 1, 5);
   LGRAPHinsertArc( G, 2, 4); LGRAPHinsertArc( G, 3, 1);
   LGRAPHinsertArc( G, 5, 3);

   MGraph H = LISTStoMATRIX( G);
   int esperado[6][6] = {{0,1,0,0,0,1},{1,0,0,0,0,1},{0,0,0,0,1,0},
                         {0,1,0,0,0,0},{0,0,0,0,0,0},{0,0,0,1,0,0}};
   int ok = (H->A == G->A);
   for (vertex v = 0; v < 6; ++v)
      for (vertex w = 0; w < 6; ++w)
         if (H->adj[v][w] != esperado[v][w]) ok = 0;
   printf( "A na matriz = %d (esperado %d)\n", H->A, G->A);
   printf( ok ? "OK: verifique sua implementacao -- passou\n"
              : "FALHOU: revise sua implementacao\n");
   return 0;
}

In [ ]:
compilar_e_rodar('ex4_conversao.c')

## 5. Grafos não-dirigidos

Um grafo é **não-dirigido** se cada arco `v-w` tem seu antiparalelo `w-v`. Cada
par de arcos antiparalelos é uma **aresta**, e portanto $E = A/2$.

Na prática, isso significa uma única mudança no código: inserir uma aresta é
inserir **dois** arcos.

### Exercício 5 — arestas e graus

Complete `UGRAPHinsertEdge(G, v, w)` (insere a aresta `v-w`) e
`UGRAPHdegrees(G, g)` (preenche `g[v]` com o grau de `v`). Em um grafo
não-dirigido, o grau de `v` é igual ao seu grau de saída — e a soma dos graus
vale $2E$.

O grafo de teste é o da aula, com arestas
`0-1  0-2  0-5  1-2  2-3  2-4  3-4  3-5`.

*(Exercício adaptado de Feofiloff, cap. Estruturas de dados para grafos, Ex. 3.)*

In [ ]:
%%writefile src/ex5_undirected.c
/* Exercicio 5: grafos nao-dirigidos com listas de adjacencia. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));
   a->w = w; a->next = next; return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return;
   G->adj[v] = NEWnode( w, G->adj[v]); G->A++;
}

/* TODO: insira a aresta v-w, ou seja, os arcos v-w e w-v. */
void UGRAPHinsertEdge( Graph G, vertex v, vertex w) {
   /* implemente aqui */
}

/* TODO: preencha g[v] com o grau do vertice v. */
void UGRAPHdegrees( Graph G, int g[]) {
   /* implemente aqui */
}

int main( void) {
   Graph G = GRAPHinit( 6);
   UGRAPHinsertEdge( G, 0, 1); UGRAPHinsertEdge( G, 0, 2);
   UGRAPHinsertEdge( G, 0, 5); UGRAPHinsertEdge( G, 1, 2);
   UGRAPHinsertEdge( G, 2, 3); UGRAPHinsertEdge( G, 2, 4);
   UGRAPHinsertEdge( G, 3, 4); UGRAPHinsertEdge( G, 3, 5);

   int g[6], soma = 0, ok = 1;
   int esperado[] = {3, 2, 4, 3, 2, 2};
   UGRAPHdegrees( G, g);
   for (vertex v = 0; v < 6; ++v) {
      printf( "grau[%d] = %d (esperado %d)\n", v, g[v], esperado[v]);
      if (g[v] != esperado[v]) ok = 0;
      soma += g[v];
   }
   printf( "A = %d (esperado 16), E = %d (esperado 8), soma dos graus = %d\n",
           G->A, G->A / 2, soma);
   printf( (ok && G->A == 16 && soma == 2 * (G->A / 2))
           ? "OK: verifique sua implementacao -- passou\n"
           : "FALHOU: revise sua implementacao\n");
   return 0;
}

In [ ]:
compilar_e_rodar('ex5_undirected.c')

## Desafio Final — leitura de um arquivo de arcos e escolha da representação

Um **arquivo de arcos** é um arquivo de texto em que a primeira linha contém
$V$, a segunda contém $A$, e cada uma das $A$ linhas seguintes contém dois
inteiros em $0..V-1$ que descrevem um arco.

Escreva um programa que:

1. leia um arquivo de arcos da entrada padrão;
2. construa o grafo usando `GRAPHinit()` e `GRAPHinsertArc()`;
3. imprima $V$, $A$ e o tamanho $V+A$ do grafo;
4. imprima o vetor de listas de adjacência;
5. imprima as fontes e os sorvedouros;
6. imprima quantas células uma **matriz** de adjacências gastaria ($V^2$) e
   quantos nós as **listas** gastam ($A$), e conclua qual representação é
   preferível para esse grafo.

Use como entrada o arquivo gerado na célula seguinte, que descreve o grafo do
Exemplo A do capítulo *Grafos* (Feofiloff): 12 vértices, 16 arcos — um grafo
claramente esparso.

In [ ]:
# Arquivo de arcos do Exemplo A: 0-5 0-6 2-0 2-3 3-6 3-10 4-1 5-2 5-10
#                                6-2 7-8 7-11 8-1 8-4 10-3 11-8
arcos = [(0,5),(0,6),(2,0),(2,3),(3,6),(3,10),(4,1),(5,2),
         (5,10),(6,2),(7,8),(7,11),(8,1),(8,4),(10,3),(11,8)]
with open('src/exemploA.txt', 'w') as f:
    f.write('12\n%d\n' % len(arcos))
    for v, w in arcos:
        f.write('%d %d\n' % (v, w))
print(open('src/exemploA.txt').read())

In [ ]:
%%writefile src/desafio.c
/* Desafio final: leitura de um arquivo de arcos da entrada padrao. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));
   a->w = w; a->next = next; return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return;
   G->adj[v] = NEWnode( w, G->adj[v]); G->A++;
}

/* TODO: leia V, depois A, depois os A pares de vertices, e devolva o grafo. */
Graph GRAPHinputArcs( void) {
   /* implemente aqui */
   return NULL;
}

int main( void) {
   Graph G = GRAPHinputArcs();
   if (G == NULL) { printf( "GRAPHinputArcs() ainda nao implementada\n"); return 1; }

   /* TODO: itens 3 a 6 do enunciado. */

   return 0;
}

In [ ]:
compilar_e_rodar('desafio.c', entrada=open('src/exemploA.txt').read())

### Sua conclusão

Responda aqui (edite esta célula):

- Para o grafo do Exemplo A ($V = 12$, $A = 16$), a matriz gastaria ____ células
  e as listas gastam ____ nós.
- Qual representação você escolheria e por quê?
- Para qual operação a matriz continuaria sendo a melhor escolha, mesmo neste
  grafo esparso?

## Referências

Este tutorial segue o material e a notação de:

- **P. Feofiloff.** *Algoritmos para Grafos (em linguagem C)*. DCC/IME-USP, 2020.
  Capítulos [Grafos](https://www.ime.usp.br/~pf/algoritmos_para_grafos/aulas/graphs.html)
  e [Estruturas de dados para grafos](https://www.ime.usp.br/~pf/algoritmos_para_grafos/aulas/graphdatastructs.html).
  Disponível em <https://www.ime.usp.br/~pf/algoritmos_para_grafos/>.
- **R. Sedgewick.** *Algorithms in C, Part 5: Graph Algorithms*. 3. ed.
  Addison-Wesley, 2002. (Fonte dos exemplos e exercícios do Cap. 17 citados por Feofiloff.)
- **T. H. Cormen, C. E. Leiserson, R. L. Rivest, C. Stein.** *Introduction to
  Algorithms*. 3. ed. MIT Press, 2009. Seção 22.1.

A lista completa está em `../referencias.bib`.